# SQL Queries: Does Outside Money Win Elections?
## Super PAC Spending and Electoral Outcomes in 2024 U.S. House Races

This notebook contains all 10 required SQL queries for the project. Each query is heavily commented to explain what it does, how it works, and what the output revealed. All queries run against a SQLite database (`superpac.db`) built from three datasets:
- **`main`** — merged candidate + Super PAC + election results table
- **`candidates`** — FEC candidate summary with incumbency and spending
- **`districts`** — district-level election results with competitiveness classification
- **`superpac_spending`** — aggregated Super PAC spending by candidate

SQL types used: GROUP BY, JOIN (3 queries), Window Functions (2 queries), Subqueries (2 queries).

## Setup

In [1]:
import pandas as pd
import sqlite3

conn = sqlite3.connect("superpac.db")
print("Connected to superpac.db")


cursor = conn.cursor()
cursor.execute("SELECT name FROM sqlite_master WHERE type='table'")
print("Tables:", cursor.fetchall())

Connected to superpac.db
Tables: [('main',), ('districts',), ('superpac_spending',), ('candidates',)]


## Query 1: Total and Average Super PAC Spending by Party
**Type:** GROUP BY

**What it does:** Groups all candidates by party affiliation (Democrat or Republican) and calculates the total and average Super PAC spending received per party.

**How it works:** Uses `SUM` and `AVG` aggregation functions on `superpac_total`, filtered to only major party candidates. Results are rounded to 2 decimal places.

**Why we ran it:** This is our baseline comparison. Before examining competitiveness or outcomes, we need to understand the overall scale and distribution of Super PAC spending across the two major parties.

**Output summary:** Democrats (1,329 candidates) received a total of 182M with an average of 137K per candidate. Republicans (1,569 candidates) received 161M with an average of 103K. The distribution within each party is highly skewed — a small number of high-profile races absorbed the majority of outside money.

In [2]:
q1 = pd.read_sql_query("""
    -- Query 1: Total and average Super PAC spending by party
    -- Groups by party to compare overall scale of outside spending
    SELECT 
        CAND_PTY_AFFILIATION,
        COUNT(*) AS num_candidates,
        ROUND(SUM(superpac_total), 2) AS total_superpac,
        ROUND(AVG(superpac_total), 2) AS avg_superpac
    FROM candidates
    WHERE CAND_PTY_AFFILIATION IN ('DEM', 'REP')
    GROUP BY CAND_PTY_AFFILIATION
""", conn)
display(q1)

,CAND_PTY_AFFILIATION,num_candidates,total_superpac,avg_superpac
0,DEM,1329,182140390.0,137050.71
1,REP,1569,161250579.0,102772.84


## Query 2: Average Super PAC Spending by District Competitiveness
**Type:** GROUP BY

**What it does:** Groups districts by their competitiveness category (Toss-up, Competitive, Likely Safe, Safe) and calculates average and total Super PAC spending per group.

**How it works:** Uses `COUNT(DISTINCT ...)` to count unique districts rather than candidates, and `AVG` and `SUM` on `superpac_total`. Results are ordered by average spending descending.

**Why we ran it:** This directly addresses our third research question — do Super PACs concentrate money in competitive races? By grouping districts into competitiveness tiers, we can see whether outside money follows electoral uncertainty.

**Output summary:** Toss-up districts received an average of 486K per candidate — nearly 5x more than Competitive (98K), Likely Safe (48K), or Safe (69K) districts. This strongly confirms that Super PAC money flows to the most contested races.

In [3]:
q2 = pd.read_sql_query("""
    -- Query 2: Average Super PAC spending by district competitiveness
    -- Tests whether more money flows into competitive vs safe districts
    SELECT 
        competitiveness,
        COUNT(DISTINCT CAND_OFFICE_ST || CAND_OFFICE_DISTRICT) AS num_districts,
        ROUND(AVG(superpac_total), 2) AS avg_superpac,
        ROUND(SUM(superpac_total), 2) AS total_superpac
    FROM main
    WHERE CAND_PTY_AFFILIATION IN ('DEM', 'REP')
    GROUP BY competitiveness
    ORDER BY avg_superpac DESC
""", conn)
display(q2)

,competitiveness,num_districts,avg_superpac,total_superpac
0,Toss-up,38,486711.94,162561788.0
1,Competitive,71,97913.38,59433424.0
2,Safe,205,69639.14,78761869.0
3,Likely Safe,122,48273.28,37894527.0


## Query 3: Top 10 Districts by Total Super PAC Spending
**Type:** GROUP BY + ORDER BY + LIMIT

**What it does:** Aggregates total Super PAC spending at the district level, ranks districts by total spending, and returns the top 10 with their competitiveness classification and winning party.

**How it works:** Groups by state and district, sums `superpac_total`, and orders descending. `LIMIT 10` returns only the top spenders.

**Why we ran it:** Identifying the highest-spending districts helps us understand where Super PACs concentrated resources most heavily and whether those districts were indeed competitive.

**Output summary:** The top 10 were dominated by Toss-up races. New York's 16th district topped the list at 16.8M, unusually high even by competitive race standards, driven by a high-profile primary challenge. Alaska's at-large district came second at 15.7M.

In [4]:
q3 = pd.read_sql_query("""
    -- Query 3: Top 10 districts by total Super PAC spending
    -- Identifies which races attracted the most outside money nationally
    SELECT 
        CAND_OFFICE_ST,
        CAND_OFFICE_DISTRICT,
        competitiveness,
        winner_party,
        ROUND(SUM(superpac_total), 2) AS total_superpac
    FROM main
    WHERE CAND_PTY_AFFILIATION IN ('DEM', 'REP')
    GROUP BY CAND_OFFICE_ST, CAND_OFFICE_DISTRICT
    ORDER BY total_superpac DESC
    LIMIT 10
""", conn)
display(q3)

,CAND_OFFICE_ST,CAND_OFFICE_DISTRICT,competitiveness,winner_party,total_superpac
0,NY,16.0,Safe,DEMOCRAT,16838475.0
1,AK,0.0,Toss-up,REPUBLICAN,15764631.0
2,CA,47.0,Toss-up,DEMOCRAT,14142592.0
3,NY,19.0,Toss-up,DEMOCRAT,11607172.0
4,MO,1.0,Safe,DEMOCRAT,11602210.0
5,CA,45.0,Toss-up,DEMOCRAT,8733395.0
6,CO,8.0,Toss-up,REPUBLICAN,8283007.0
7,SC,1.0,Likely Safe,REPUBLICAN,7382003.0
8,NE,2.0,Toss-up,REPUBLICAN,6994673.0
9,VA,7.0,Toss-up,DEMOCRAT,6920032.0


## Query 4: Super PAC Spending for Winners vs Losers
**Type:** JOIN (CASE-based outcome join)

**What it does:** Classifies each candidate as Winner or Loser by matching their party affiliation against the winning party in their district. Calculates average Super PAC spending for/against and average own spending, grouped by party and outcome.

**How it works:** Uses a `CASE` statement to determine outcome: if a Democrat's party matches `winner_party = 'DEMOCRAT'` they are a Winner, otherwise a Loser. Groups by both party and outcome.

**Why we ran it:** This connects candidate-level spending to electoral outcomes. It directly addresses our first research question: is Super PAC money associated with winning?

**Output summary:** Winners received roughly twice as much Super PAC support as losers across both parties. Democratic winners averaged 111K vs 45K for losing Democrats. Republican winners averaged 70K vs 31K for losers. Notably, the gap in candidates' own spending was even larger, suggesting self-funding matters more than outside support.

In [13]:
q4 = pd.read_sql_query("""
    -- Query 4: Super PAC spending for winners vs losers
    -- JOIN-style: uses CASE to connect candidate party with district winner
    -- to classify each candidate as Winner or Loser
    
    SELECT
        m.CAND_PTY_AFFILIATION,
        d.competitiveness,
        CASE
            WHEN (m.CAND_PTY_AFFILIATION = 'DEM' AND m.winner_party = 'DEMOCRAT')
              OR (m.CAND_PTY_AFFILIATION = 'REP' AND m.winner_party = 'REPUBLICAN')
            THEN 'Winner'
            ELSE 'Loser'
        END AS outcome,
        COUNT(*) AS num_candidates,
        ROUND(AVG(m.superpac_for), 2) AS avg_superpac_for,
        ROUND(AVG(m.superpac_against), 2) AS avg_superpac_against,
        ROUND(AVG(m.TTL_DISB), 2) AS avg_candidate_spending
    FROM main m
    JOIN districts d
        ON m.CAND_OFFICE_ST = d.CAND_OFFICE_ST
       AND m.CAND_OFFICE_DISTRICT = d.CAND_OFFICE_DISTRICT
    WHERE m.CAND_PTY_AFFILIATION IN ('DEM', 'REP')
    GROUP BY m.CAND_PTY_AFFILIATION, d.competitiveness, outcome
    ORDER BY m.CAND_PTY_AFFILIATION, outcome
""", conn)

print("\n Super PAC spending for winners vs losers")
display(q4)


 Super PAC spending for winners vs losers


,CAND_PTY_AFFILIATION,competitiveness,outcome,num_candidates,avg_superpac_for,avg_superpac_against,avg_candidate_spending
0,DEM,Competitive,Loser,102,32654.72,24844.53,670597.13
1,DEM,Likely Safe,Loser,204,1034.32,21.17,145713.42
2,DEM,Safe,Loser,197,31.35,507.90,139279.88
3,DEM,Toss-up,Loser,76,296250.14,268415.47,2209051.09
4,DEM,Competitive,Winner,162,132996.81,18829.51,1628036.56
5,DEM,Likely Safe,Winner,142,52942.19,6595.63,795021.24
6,DEM,Safe,Winner,341,64674.30,79875.94,1125495.86
7,DEM,Toss-up,Winner,83,358126.53,252686.25,2507692.57
8,REP,Competitive,Loser,235,12855.50,12338.09,309168.49
9,REP,Likely Safe,Loser,146,752.08,0.00,77864.89


## Query 5: District Ranking by Super PAC Spending Within State
**Type:** Window Function (RANK + PARTITION BY)

**What it does:** Ranks each district within its state by total Super PAC spending, from highest to lowest. Shows which district in each state attracted the most outside money.

**How it works:** Uses `RANK() OVER (PARTITION BY CAND_OFFICE_ST ORDER BY SUM(superpac_total) DESC)` — the `PARTITION BY` resets the ranking for each state, so each state's top district gets rank 1.

**Why we ran it:** A national ranking alone would obscure geographic concentration. Partitioning within states reveals whether Super PAC money is spread evenly across a state's districts or concentrated in one or two races.

**Output summary:** In most states, one district dominates — absorbing nearly all outside money while safe seats receive almost nothing. In Arizona, the top two districts (both Toss-ups) absorbed 70% of all Super PAC spending in the state.

In [6]:
q5 = pd.read_sql_query("""
    -- Query 5: Window function - rank districts by Super PAC spending within state
    -- PARTITION BY state resets rank for each state independently
    -- Shows geographic concentration of outside money within states
    SELECT 
        CAND_OFFICE_ST,
        CAND_OFFICE_DISTRICT,
        competitiveness,
        ROUND(SUM(superpac_total), 2) AS total_superpac,
        RANK() OVER (
            PARTITION BY CAND_OFFICE_ST 
            ORDER BY SUM(superpac_total) DESC
        ) AS rank_in_state
    FROM main
    WHERE CAND_PTY_AFFILIATION IN ('DEM', 'REP')
    GROUP BY CAND_OFFICE_ST, CAND_OFFICE_DISTRICT
    ORDER BY CAND_OFFICE_ST, rank_in_state
    LIMIT 20
""", conn)
display(q5)

,CAND_OFFICE_ST,CAND_OFFICE_DISTRICT,competitiveness,total_superpac,rank_in_state
0,AK,0.0,Toss-up,15764631.0,1
1,AL,2.0,Competitive,3124498.0,1
2,AL,1.0,Safe,2731059.0,2
3,AL,7.0,Likely Safe,75926.0,3
4,AL,6.0,Safe,0.0,4
5,AL,5.0,Safe,0.0,4
6,AL,4.0,Safe,0.0,4
7,AL,3.0,Safe,0.0,4
8,AR,3.0,Safe,188992.0,1
9,AR,2.0,Likely Safe,156346.0,2


## Query 6: Share of Competitive Districts Among High vs Low Spending
**Type:** Subquery (nested)

**What it does:** Classifies each district as above or below average Super PAC spending, then calculates the percentage of competitive districts within each group.

**How it works:** The inner subquery computes district-level totals and compares them to the national average using a second nested subquery (`SELECT AVG(...) FROM main`). The outer query then counts and calculates the percentage of competitive districts in each group.

**Why we ran it:** This tests whether high-spending districts are systematically more competitive — directly addressing our research question about whether Super PAC money targets contested races.

**Output summary:** Among districts with above-average Super PAC spending, 56% were competitive. Among those with below-average spending, only 12% were competitive. This five-fold difference strongly confirms that Super PAC targeting is strategic.

In [7]:
q6 = pd.read_sql_query("""
    -- Query 6: Subquery - are high-spending districts more competitive?
    -- Outer query: counts competitive districts per spending group
    -- Inner subquery: classifies districts as above/below average spending
    -- Innermost subquery: computes national average Super PAC spending
    SELECT 
        group_label,
        COUNT(*) AS num_districts,
        SUM(CASE WHEN competitiveness IN ('Toss-up', 'Competitive') 
            THEN 1 ELSE 0 END) AS num_competitive,
        ROUND(100.0 * SUM(CASE WHEN competitiveness IN ('Toss-up', 'Competitive') 
            THEN 1 ELSE 0 END) / COUNT(*), 1) AS pct_competitive
    FROM (
        SELECT 
            CAND_OFFICE_ST,
            CAND_OFFICE_DISTRICT,
            competitiveness,
            SUM(superpac_total) AS total_superpac,
            CASE 
                WHEN SUM(superpac_total) > (
                    SELECT AVG(superpac_total) FROM main
                    WHERE CAND_PTY_AFFILIATION IN ('DEM', 'REP')
                ) THEN 'Above average spending'
                ELSE 'Below average spending'
            END AS group_label
        FROM main
        WHERE CAND_PTY_AFFILIATION IN ('DEM', 'REP')
        GROUP BY CAND_OFFICE_ST, CAND_OFFICE_DISTRICT
    )
    GROUP BY group_label
""", conn)
display(q6)

,group_label,num_districts,num_competitive,pct_competitive
0,Above average spending,130,73,56.2
1,Below average spending,306,36,11.8


## Query 7: Each District's Share of State Super PAC Spending
**Type:** Window Function (SUM OVER PARTITION BY)

**What it does:** Calculates each district's Super PAC spending as a percentage of its state's total Super PAC spending.

**How it works:** Uses `SUM(superpac_total) * 100.0 / SUM(SUM(superpac_total)) OVER (PARTITION BY CAND_OFFICE_ST)` — the window function computes the state total without collapsing rows, allowing us to divide each district's spending by its state's total.

**Why we ran it:** Knowing one state receives more money than another doesn't tell us whether that money is spread evenly or concentrated in one race. This window function reveals within-state concentration patterns.

**Output summary:** In most states, one or two districts absorb nearly all outside money. In Alabama, two districts split 99% of all Super PAC dollars. In Arkansas, the same pattern held. Safe districts in most states received 0%.

In [8]:
q7 = pd.read_sql_query("""
    -- Query 7: Window function - each district's share of state Super PAC spending
    -- SUM() OVER (PARTITION BY state) computes state total without collapsing rows
    -- Dividing district spending by state total gives within-state concentration
    SELECT 
        CAND_OFFICE_ST,
        CAND_OFFICE_DISTRICT,
        competitiveness,
        ROUND(SUM(superpac_total), 2) AS district_superpac,
        ROUND(SUM(superpac_total) * 100.0 / SUM(SUM(superpac_total)) OVER (
            PARTITION BY CAND_OFFICE_ST
        ), 1) AS pct_of_state_total
    FROM main
    WHERE CAND_PTY_AFFILIATION IN ('DEM', 'REP')
    GROUP BY CAND_OFFICE_ST, CAND_OFFICE_DISTRICT, competitiveness
    ORDER BY CAND_OFFICE_ST, district_superpac DESC
    LIMIT 20
""", conn)
display(q7)

,CAND_OFFICE_ST,CAND_OFFICE_DISTRICT,competitiveness,district_superpac,pct_of_state_total
0,AK,0.0,Toss-up,15764631.0,100.0
1,AL,2.0,Competitive,3124498.0,52.7
2,AL,1.0,Safe,2731059.0,46.0
3,AL,7.0,Likely Safe,75926.0,1.3
4,AL,3.0,Safe,0.0,0.0
5,AL,4.0,Safe,0.0,0.0
6,AL,5.0,Safe,0.0,0.0
7,AL,6.0,Safe,0.0,0.0
8,AR,3.0,Safe,188992.0,54.5
9,AR,2.0,Likely Safe,156346.0,45.1


## Query 8: Super PAC vs Candidate Own Spending by Party and Competitiveness
**Type:** JOIN (explicit JOIN between main and districts tables)

**What it does:** Joins the `main` candidate table with the `districts` table on state and district to combine candidate-level spending data with district-level competitiveness. Calculates average Super PAC spending, average own spending, and Super PAC as a percentage of own spending, by party and competitiveness.

**How it works:** Uses an explicit `JOIN ... ON` to match candidates to their district's competitiveness classification. `NULLIF` prevents division by zero when computing the ratio.

**Why we ran it:** This addresses our second research question — is Super PAC money more important than candidates' own spending? Comparing both side-by-side across competitiveness levels and parties reveals their relative scale and strategic importance.

**Output summary:** In Toss-up races, Super PAC spending represented 25% of Democratic candidates' own disbursements and 35% of Republicans' — Republicans are more relatively dependent on outside money. Both parties concentrate Super PAC money heavily in Toss-up races.

In [9]:
q8 = pd.read_sql_query("""
    -- Query 8: JOIN - Super PAC vs candidate own spending by party and competitiveness
    -- Joins main (candidate-level) with districts (district-level competitiveness)
    -- on state + district to get competitiveness for each candidate
    -- NULLIF prevents division by zero in the ratio calculation
    SELECT 
        m.CAND_PTY_AFFILIATION,
        d.competitiveness,
        COUNT(DISTINCT m.CAND_OFFICE_ST || m.CAND_OFFICE_DISTRICT) AS num_districts,
        ROUND(AVG(m.superpac_total), 2) AS avg_superpac,
        ROUND(AVG(m.TTL_DISB), 2) AS avg_candidate_spending,
        ROUND(AVG(m.superpac_total) / NULLIF(AVG(m.TTL_DISB), 0) * 100, 1) AS superpac_as_pct_of_candidate
    FROM main m
    JOIN districts d 
        ON m.CAND_OFFICE_ST = d.CAND_OFFICE_ST 
        AND m.CAND_OFFICE_DISTRICT = d.CAND_OFFICE_DISTRICT
    WHERE m.CAND_PTY_AFFILIATION IN ('DEM', 'REP')
    GROUP BY m.CAND_PTY_AFFILIATION, d.competitiveness
    ORDER BY m.CAND_PTY_AFFILIATION, avg_superpac DESC
""", conn)
display(q8)

,CAND_PTY_AFFILIATION,competitiveness,num_districts,avg_superpac,avg_candidate_spending,superpac_as_pct_of_candidate
0,DEM,Toss-up,38,588755.02,2364945.70,24.9
1,DEM,Competitive,71,115381.77,1258116.78,9.2
2,DEM,Safe,196,91817.59,764372.17,12.0
3,DEM,Likely Safe,120,25056.91,412192.35,6.1
4,REP,Toss-up,38,393998.51,1134464.49,34.7
5,REP,Competitive,71,84468.33,547561.13,15.4
6,REP,Likely Safe,122,66571.38,524324.55,12.7
7,REP,Safe,191,49517.72,595883.02,8.3


## Query 9: Win Rate Among Candidates with Negative Super PAC Spending
**Type:** Subquery (inline view)

**What it does:** Filters to candidates who had any Super PAC money spent against them, classifies them as Won or Lost, and compares their average spending-against and own campaign spending by outcome.

**How it works:** The inner subquery creates a derived table with only candidates where `superpac_against > 0`, adding a Won/Lost classification using a CASE statement. The outer query aggregates by outcome.

**Why we ran it:** We wanted to test whether being targeted by opposition Super PAC spending reliably predicts losing — a key question about the effectiveness of negative outside money.

**Output summary:** Surprisingly, candidates with Super PAC money spent against them won three times more often than they lost (155 wins vs 55 losses). The average amount spent against winners and losers was nearly identical (713K each). This suggests strong candidates attract opposition spending precisely because they are competitive.

In [10]:
q9 = pd.read_sql_query("""
    -- Query 9: Subquery - win rate among candidates with negative Super PAC spending
    -- Inner subquery: filters to candidates with superpac_against > 0
    --   and classifies each as Won or Lost based on party-outcome matching
    -- Outer query: aggregates by outcome to compare spending levels
    SELECT 
        outcome,
        COUNT(*) AS num_candidates,
        ROUND(AVG(superpac_against), 2) AS avg_superpac_against,
        ROUND(AVG(TTL_DISB), 2) AS avg_candidate_spending
    FROM (
        SELECT 
            CAND_PTY_AFFILIATION,
            superpac_against,
            TTL_DISB,
            CASE 
                WHEN CAND_PTY_AFFILIATION = 'DEM' AND winner_party = 'DEMOCRAT' THEN 'Won'
                WHEN CAND_PTY_AFFILIATION = 'REP' AND winner_party = 'REPUBLICAN' THEN 'Won'
                ELSE 'Lost'
            END AS outcome
        FROM main
        WHERE CAND_PTY_AFFILIATION IN ('DEM', 'REP')
            AND superpac_against > 0
    )
    GROUP BY outcome
    ORDER BY outcome
""", conn)
display(q9)

,outcome,num_candidates,avg_superpac_against,avg_candidate_spending
0,Lost,55,713383.89,4145473.04
1,Won,155,713245.03,3740460.11


## Query 10: Does Super PAC Support Replace or Complement Candidate Spending?
**Type:** JOIN + Subquery (nested subquery within CASE)

**What it does:** Classifies candidates into three tiers based on their Super PAC support level: High (above average), Low (above zero but below average), and None. Then, it calculates average own spending and average Super PAC spending per tier.

**How it works:** The inner subquery creates a derived table with a CASE statement that uses a nested subquery to compute the average Super PAC total as a threshold for classification. The outer query aggregates by support level. `NULLIF` prevents division by zero in the ratio.

**Why we ran it:** A key question in campaign finance research is whether outside money substitutes for or adds to candidate fundraising. If candidates with Super PAC support raise less themselves, outside money replaces their effort. If they raise more, it piles on top.

**Output summary:** Candidates with high Super PAC support spent an average of 4.5M of their own money (nearly double the 2.3M spent by those with low support, and dramatically more than the 228K spent by those with none). Super PAC money clearly complements rather than replaces a strong campaign.

In [11]:
q10 = pd.read_sql_query("""
    -- Query 10: Subquery + JOIN - does Super PAC support replace or complement spending?
    -- Inner subquery: classifies candidates by Super PAC support level
    --   using a nested subquery to compute the average threshold
    -- Outer query: calculates average own spending and Super PAC per group
    -- NULLIF prevents division by zero in the ratio
    SELECT 
        superpac_support_level,
        COUNT(*) AS num_candidates,
        ROUND(AVG(TTL_DISB), 2) AS avg_candidate_spending,
        ROUND(AVG(clean_superpac_total), 2) AS avg_superpac,
        ROUND(AVG(TTL_DISB) / NULLIF(AVG(clean_superpac_total), 0), 2) AS candidate_to_superpac_ratio
    FROM (
        SELECT 
            CAND_NAME,
            TTL_DISB,
            superpac_total,
            CASE 
                WHEN superpac_total < 0 THEN 0
                ELSE superpac_total
            END AS clean_superpac_total,
            CASE 
                WHEN superpac_total > (
                    SELECT AVG(superpac_total) FROM main 
                    WHERE CAND_PTY_AFFILIATION IN ('DEM', 'REP')
                    AND superpac_total > 0
                ) THEN 'High Super PAC support'
                WHEN superpac_total > 0 THEN 'Low Super PAC support'
                ELSE 'No Super PAC support'
            END AS superpac_support_level
        FROM main
        WHERE CAND_PTY_AFFILIATION IN ('DEM', 'REP')
    )
    GROUP BY superpac_support_level
    ORDER BY avg_superpac DESC
""", conn)
display(q10)

,superpac_support_level,num_candidates,avg_candidate_spending,avg_superpac,candidate_to_superpac_ratio
0,High Super PAC support,137,4507559.34,2245989.85,2.01
1,Low Super PAC support,488,2263514.80,63509.17,35.64
2,No Super PAC support,2232,228392.02,0.00,NaN
